In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import itertools
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from scipy import stats
import seaborn as sns
import dask
import dask.dataframe as dd
from dask.distributed import Client
from dask.distributed import LocalCluster
from shapely.geometry import Point
import geopandas as gpd
import shapefile  # PyShp
from shapely.geometry import shape

In [2]:
df = dd.read_csv('combined_snow_water.csv')

In [3]:
# Read the shapefile using PyShp
sf = shapefile.Reader("cb_2018_us_state_20m.shp")

# Extract shapes and attributes
shapes = [shape(record.shape.__geo_interface__) for record in sf.shapeRecords()]
records = [record.record for record in sf.shapeRecords()]
columns = [field[0] for field in sf.fields[1:]]  # Skip 'DeletionFlag'

# Create a GeoDataFrame manually
us_states = gpd.GeoDataFrame(records, columns=columns, geometry=shapes)

# Exclude Alaska
us_states = us_states[~us_states['STUSPS'].isin(['AK'])]

In [ ]:
# Create geometry directly in the Dask DataFrame using map_partitions
df['geometry'] = df.map_partitions(
    lambda part: part.apply(lambda row: Point(row['X'], row['Y']), axis=1),
    meta=('geometry', 'object')
)

# Perform computation lazily (you can choose to persist or compute in smaller steps)
# gdf = df.compute()  # Don't use this yet, we need to optimize computation first

# Save intermediate results or filter before computing full DataFrame
df.persist()  # Keeps data in memory for faster computation on subsequent operations

# Optionally, limit to a smaller subset for testing (if dataset is very large)
df_sample = df.head(1000)  # Grab a small sample for testing (avoid computing full df)

# Create GeoDataFrame when ready
gdf = gpd.GeoDataFrame(df_sample.compute(), geometry='geometry')

# Set CRS once GeoDataFrame is materialized
gdf.set_crs(epsg=4326, inplace=True)

# Plot or perform operations on the GeoDataFrame
gdf.plot()